# Cluster Summary

**EmbeddingGemma 2 version** of `main/code/02_cluster_summary.ipynb`: the fine-tuned (DoubleMLDeep) product embeddings are replaced by [EmbeddingGemma 2](https://huggingface.co/google/embeddinggemma-2) embeddings (text + image, Matryoshka 256 dimensions). Datasets are read from `gemma/data`, raw data from `main/data`, outputs are written to `gemma/outputs`.

Requires the outputs of `04_evaluation.ipynb` (`sorted_effects.csv`); the cluster labels are the short versions of the headlines in `02_cluster_centroid_products.ipynb`.

In [ ]:
import numpy as np
import pandas as pd

df_val = pd.read_csv("../data/dataset_txt_only_False_embeddings_True_val.zip")
#df_train = pd.read_csv("../data/dataset_txt_only_False_embeddings_True_train.zip")
df_effects = pd.read_csv("../outputs/04_evaluation/sorted_effects.csv")

df_full = df_val # pd.concat([df_val, df_train], ignore_index=True)
df_full.head()

In [ ]:
df_effects

In [ ]:
df_val.set_index(["ASIN", "date"]).join(df_effects.set_index(["ASIN", "date"]), how="left")

In [ ]:
len(df_val.set_index(["ASIN", "date"]).join(df_effects.set_index(["ASIN", "date"]), how="left").dropna(subset=["SortEff_deep_dml"]))

In [ ]:
sim_cols = [
 'similarity_cluster_0',
 'similarity_cluster_1',
 'similarity_cluster_2',
 'similarity_cluster_3',
 'similarity_cluster_4',
]

In [ ]:
np.exp(10.685092)

In [ ]:
# assign cluster label for each row based on the highest similarity score in sim_cols

df_full['cluster_label'] = df_full[sim_cols].idxmax(axis=1)
df_full['cluster_label'] = df_full['cluster_label'].str.replace('similarity_cluster_', '').astype(int)
df_full['SR'] = np.exp(-df_full["Q_t"])
df_full['Price'] = np.exp(df_full["P_bb_t"])
df_full.head()


In [ ]:
import os
import numpy as np


# Short labels from the LLM headline exercise (used as readable cluster names)
cluster_headlines_short = {
    0: "Toddler Construction Trucks",
    1: "1:64 Die-Cast Collectibles",
    2: "Wooden Fingerboards",
    3: "Toy Train Playsets",
    4: "Light & Sound Pull-Back Cars",
}


# ── Load per-observation deep-DML elasticity and merge on (ASIN, date) ──────
sorted_eff = pd.read_csv("../outputs/04_evaluation/sorted_effects.csv")
sorted_eff["date"] = sorted_eff["date"].astype(str)
df_full["date"] = df_full["date"].astype(str)

df_char = df_full.merge(
    sorted_eff[["ASIN", "date", "SortEff_deep_dml"]],
    on=["ASIN", "date"],
    how="left",
).dropna(subset=["SortEff_deep_dml"])  # keep only rows with elasticity estimates

# ── Build per-cluster statistics ─────────────────────────────────────────────
rows = []
for c in range(5):
    sub = df_char[df_char["cluster_label"] == c]
    sub_asin = sub.groupby("ASIN").first().reset_index()  # one row per product

    n_prod         = int(sub["ASIN"].nunique())
    n_obs          = len(sub)
    mean_price     = np.exp(sub["P_bb_t"]).mean()
    median_SR      = int(np.exp(-sub_asin["Q_t"]).median())
    mean_rating    = sub["RATING"].mean()
    # Product-level median review count (avoids repeating the same product
    # across many time periods inflating the statistic)
    median_reviews = int(sub_asin["REVIEW_COUNT"].median())

    # Top-2 subcategories by number of distinct products
    top_cats = (
        sub_asin["subcat_aggregated"]
        .value_counts()
        .head(2)
        .index.tolist()
    )
    cat_str = " / ".join(top_cats)

    # Mean elasticity (observations matched to sorted_effects only)
    mean_elas = sub["SortEff_deep_dml"].mean()

    rows.append(
        {
            "Cluster": c,
            "Label": cluster_headlines_short[c],
            "# Products": n_prod,
            "# Observations": n_obs,
            "Mean Price ($)": round(mean_price, 2),
            "Median Sales Rank": median_SR,
            "Mean Rating": round(mean_rating, 2),
            "Median Reviews": median_reviews,
            "Dominant Product Type": cat_str,
            "Mean Elasticity": round(mean_elas, 3),
        }
    )

df_summary = pd.DataFrame(rows).set_index("Cluster")

# ── Save ─────────────────────────────────────────────────────────────────────
os.makedirs("../outputs/R2", exist_ok=True)
df_summary.to_csv(os.path.join("../outputs/R2", "cluster_summary_table.csv"))

# ── Display ──────────────────────────────────────────────────────────────────
df_summary.style.format(
    {
        "Mean Price ($)": "${:.2f}",
        "Mean Rating": "{:.2f}",
        "Mean Elasticity": "{:.3f}",
    }
)